# 🧠 AI Management Consultant — Multi-Agent System (OpenAI Agents SDK)

Run this notebook top-to-bottom in **Google Colab** or **VS Code (Jupyter extension)**.
No terminal needed — every stage prints a nicely formatted progress panel below its cell.

**What this notebook does:**
1. Sets up 7 specialized AI agents (Triage, Business Analyst, Market Research, Benchmarking, Financial Analysis, Strategy Advisor, Report Writer)
2. Runs them as a pipeline: handoff → parallel research → strategy synthesis → report generation
3. Pauses for **your approval** before finalizing the report
4. Generates a downloadable `.docx` consulting report

**Provider setup:** this version spreads work across **4 free API keys** (2x Groq, 1x OpenRouter, 1x Gemini) so no single
free tier gets exhausted. Each agent is pinned to a specific key — see section 4.

**Efficiency notes (read this if you're debugging quota errors):**
- No agent combines `tools=[...]` with `output_type=...` in the same call — several providers
  (notably Groq) reject that combination outright. Structured Pydantic schemas still exist (section 6)
  and are used for documentation / grading purposes, and every tool has a typed signature.
- There is **no shared conversation session** across stages. Each stage gets an explicit, self-contained
  prompt built from the shared `ConsultingContext`, instead of replaying the whole engagement's
  history (which was the real cause of token-quota exhaustion, independent of how many keys you use).
- `max_turns` is capped per stage to stop a stuck tool-call loop from burning your quota.


## 1. Install dependencies

In [4]:
%pip install -q openai-agents python-docx pydantic python-dotenv ddgs nest_asyncio
print("✅ Packages installed")

^C
Note: you may need to restart the kernel to use updated packages.
✅ Packages installed


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

## 2. Set your 4 API keys

You need **4 free API keys** — none require a credit card:

| Env var | Where to get it |
|---|---|
| `GEMINI_API_KEY` | https://aistudio.google.com/apikey |
| `GROQ_API_KEY_1` | https://console.groq.com/keys |
| `GROQ_API_KEY_2` | a second Groq account/key — https://console.groq.com/keys |
| `OPENROUTER_API_KEY` | https://openrouter.ai/keys |

**In Colab:** click the 🔑 icon in the left sidebar → "Secrets" → add each key, then toggle "Notebook access" on.
**In VS Code / locally:** falls back to a secure password prompt for any key not found as an env var.


In [ ]:
import os
import getpass

def get_secret_key(key_name: str, prompt_msg: str) -> str:
    """
    Safely retrieves an API key from:
    1. Google Colab Secrets (userdata)
    2. System Environment Variables
    3. Interactive User Prompt (getpass)
    """
    try:
        from google.colab import userdata
        key = userdata.get(key_name)
        if key:
            return key
    except Exception:
        pass

    if os.environ.get(key_name):
        return os.environ[key_name]

    return getpass.getpass(prompt_msg)

# Set all 4 keys securely
os.environ["GEMINI_API_KEY"]     = get_secret_key("GEMINI_API_KEY", "Paste your Gemini API Key (aistudio.google.com): ")
os.environ["GROQ_API_KEY_1"]     = get_secret_key("GROQ_API_KEY_1", "Paste Groq API Key #1 (console.groq.com/keys): ")
os.environ["GROQ_API_KEY_2"]     = get_secret_key("GROQ_API_KEY_2", "Paste Groq API Key #2 (secondary Groq account): ")
os.environ["OPENROUTER_API_KEY"] = get_secret_key("OPENROUTER_API_KEY", "Paste OpenRouter API Key (openrouter.ai/keys): ")

print("✅ All 4 API keys configured for this session")

✅ All 4 API keys configured for this session


## 3. Notebook display + async helpers

In [ ]:
import asyncio, nest_asyncio
nest_asyncio.apply()  # lets asyncio.run() work inside a running notebook kernel

from IPython.display import display, Markdown, HTML
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("ai_consultant")

def show(title: str, obj=None, color="#2563eb"):
    """Pretty-print a stage header + a pydantic model (or any object) as Markdown."""
    display(HTML(f"<h3 style='color:{color};border-bottom:2px solid {color};padding-bottom:4px'>▶ {title}</h3>"))
    if obj is not None:
        if hasattr(obj, "model_dump"):
            for k, v in obj.model_dump().items():
                if isinstance(v, list):
                    display(Markdown(f"**{k}:**"))
                    for item in v:
                        display(Markdown(f"- {item}"))
                else:
                    display(Markdown(f"**{k}:** {v}"))
        else:
            display(Markdown(str(obj)))

def show_error(title: str, msg: str):
    display(HTML(f"<div style=\'background:#fee2e2;border-left:4px solid #dc2626;padding:8px 12px;margin:4px 0\'>⚠️ <b>{title} failed:</b> {msg}</div>"))

print("✅ Display helpers ready")

ModuleNotFoundError: No module named 'nest_asyncio'

## 4. Model configuration — 4 providers, 4 keys

Each agent is pinned to a specific provider/key below, chosen so the 3 agents that run **in parallel**
(Market Research, Benchmarking, Financial Analysis) never share a key and can't collide on the same
per-minute rate limit.

| Agent | Provider | Key |
|---|---|---|
| Triage | Groq (fast) | `GROQ_API_KEY_1` |
| Business Analyst | OpenRouter | `OPENROUTER_API_KEY` |
| Market Research | OpenRouter | `OPENROUTER_API_KEY` |
| Benchmarking | Groq (heavy) | `GROQ_API_KEY_2` |
| Financial Analysis | Gemini | `GEMINI_API_KEY` |
| Strategy Advisor | Gemini | `GEMINI_API_KEY` |
| Report Writer | Groq (fast) | `GROQ_API_KEY_1` |


In [ ]:
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

set_tracing_disabled(True)

# --------------------------------------------------
# 1. AsyncOpenAI clients — one per provider/key
# --------------------------------------------------

groq_client_1 = AsyncOpenAI(
    api_key=os.environ["GROQ_API_KEY_1"],
    base_url="https://api.groq.com/openai/v1",
)

groq_client_2 = AsyncOpenAI(
    api_key=os.environ["GROQ_API_KEY_2"],
    base_url="https://api.groq.com/openai/v1",
)

openrouter_client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

gemini_client = AsyncOpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# --------------------------------------------------
# 2. Agents SDK model objects
# --------------------------------------------------

groq_fast_model = OpenAIChatCompletionsModel(
    model="llama-3.1-8b-instant",
    openai_client=groq_client_1,
)

groq_heavy_model = OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile",
    openai_client=groq_client_2,
)

openrouter_free_model = OpenAIChatCompletionsModel(
    model="openai/gpt-oss-20b:free",  # meta-llama's free slug was delisted by OpenRouter in Aug 2026
    openai_client=openrouter_client,
)

gemini_model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash",
    openai_client=gemini_client,
)

print("✅ 4 model objects created (groq_fast_model, groq_heavy_model, openrouter_free_model, gemini_model)")

✅ 4 model objects created (groq_fast_model, groq_heavy_model, openrouter_free_model, gemini_model)


## 5. Shared context (cross-agent memory)

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

@dataclass
class ConsultingContext:
    client_id: str
    company_name: str
    industry: str
    raw_problem_statement: str
    financial_figures: Dict[str, float] = field(default_factory=dict)
    uploaded_doc_paths: List[str] = field(default_factory=list)

    business_analysis: Optional[Any] = None
    market_research: Optional[Any] = None
    benchmark_report: Optional[Any] = None
    financial_analysis: Optional[Any] = None
    strategy: Optional[Any] = None
    report_path: Optional[str] = None
    approval_status: str = "pending"

    findings_log: List[Dict[str, str]] = field(default_factory=list)

    def log(self, agent: str, summary: str) -> None:
        self.findings_log.append({"agent": agent, "summary": summary})

print("✅ ConsultingContext defined")

✅ ConsultingContext defined


## 6. Structured outputs (Pydantic models)

These schemas document the shape of each stage's output and are used for tool argument typing.
They aren't forced through `output_type` on the research agents (see the efficiency note in the intro
cell for why), but every field a stage is asked to fill in maps 1:1 to a class below — useful for your
project write-up under "Structured outputs".

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class BusinessProblemAnalysis(BaseModel):
    industry: str
    company_stage: str = Field(description="e.g. pre-seed, growth, mature")
    core_question: str
    key_constraints: List[str]
    data_gaps: List[str]

class MarketResearch(BaseModel):
    market_size_summary: str
    growth_trend: str
    key_drivers: List[str]
    risks: List[str]
    sources: List[str]

class Competitor(BaseModel):
    name: str
    positioning: str
    strengths: List[str]
    weaknesses: List[str]

class BenchmarkReport(BaseModel):
    competitors: List[Competitor]
    company_relative_position: str

class FinancialAnalysis(BaseModel):
    gross_margin_pct: float
    net_margin_pct: float
    current_ratio: float
    runway_months: float
    headline_assessment: str

class SwotItem(BaseModel):
    category: str
    point: str

class StrategicRecommendation(BaseModel):
    title: str
    rationale: str
    priority: str
    expected_impact: str

class StrategyRecommendation(BaseModel):
    swot: List[SwotItem]
    recommendations: List[StrategicRecommendation]
    self_review_notes: str

print("✅ 6 structured output schemas defined")

✅ 6 structured output schemas defined


## 7. Tools (8 total)

Financial calculators, free web search (DuckDuckGo), competitor lookup, RAG-style knowledge-base search,
a findings logger, `.docx` report generation, and a human-approval gate.

In [ ]:
import os
from agents import function_tool, RunContextWrapper
from ddgs import DDGS

# --------------------------------------------------
# 1. knowledge_base_search — mock RAG over an in-memory
#    "uploaded documents" store (swap for a real vector
#    store / file search in production).
# --------------------------------------------------

_KNOWLEDGE_BASE = [
    "Internal memo: Northwind Bakery Co. currently operates a single retail storefront with "
    "3 full-time and 4 part-time staff. Average daily foot traffic is 180 customers.",
    "Ops note: current oven capacity supports roughly 400 units/day; wholesale expansion would "
    "require either a second oven line or a co-packing partner.",
    "Customer survey (n=210): 68% of respondents said they would buy via an online store if "
    "shipping cost stayed under $8.",
]

@function_tool
def knowledge_base_search(query: str) -> str:
    """Search the client's uploaded internal documents/knowledge base for facts relevant to the query.

    Args:
        query: The topic or question to search the knowledge base for.
    """
    query_lower = query.lower()
    hits = [doc for doc in _KNOWLEDGE_BASE if any(w in doc.lower() for w in query_lower.split())]
    if not hits:
        hits = _KNOWLEDGE_BASE  # fall back to returning everything for a small demo KB
    return "\n\n".join(f"- {h}" for h in hits)


# --------------------------------------------------
# 2. web_search — free DuckDuckGo search
# --------------------------------------------------

@function_tool
def web_search(query: str) -> str:
    """Search the live web for current information (market data, news, competitor info, etc).

    Args:
        query: The search query.
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return f"No web results found for: {query}"
        return "\n\n".join(f"- {r.get('title','')}: {r.get('body','')} ({r.get('href','')})" for r in results)
    except Exception as e:
        return f"Web search failed ({e}). Proceed using general knowledge and flag this as a data gap."


# --------------------------------------------------
# 3. competitor_benchmark_lookup — starter competitor
#    archetypes by industry keyword (swap for a real
#    market-intelligence API in production).
# --------------------------------------------------

_COMPETITOR_ARCHETYPES = {
    "bakery": ["a large regional wholesale bakery chain", "a direct-to-consumer e-commerce bakery brand",
               "a local artisan competitor with a loyal following"],
    "retail": ["a national big-box retailer", "a fast-growing DTC e-commerce brand",
               "a regional specialty chain"],
    "saas": ["an established enterprise incumbent", "a fast-moving venture-backed challenger",
              "an open-source / self-hosted alternative"],
}

@function_tool
def competitor_benchmark_lookup(industry: str) -> str:
    """Return a starter list of typical competitor archetypes for the given industry, to be
    refined with web_search into real, named competitors.

    Args:
        industry: The client's industry, e.g. "artisan bakery / specialty food retail".
    """
    key = next((k for k in _COMPETITOR_ARCHETYPES if k in industry.lower()), None)
    archetypes = _COMPETITOR_ARCHETYPES.get(key, ["an established market leader", "a fast-growing challenger",
                                                     "a low-cost alternative provider"])
    return ("Typical competitor archetypes for this industry (use web_search to find real, named "
            "companies matching these):\n" + "\n".join(f"- {a}" for a in archetypes))


# --------------------------------------------------
# 4 & 5. Financial calculators
# --------------------------------------------------

@function_tool
def financial_ratio_calculator(revenue: float, cogs: float, net_income: float,
                                current_assets: float, current_liabilities: float) -> str:
    """Compute gross margin %, net margin %, and current ratio from the given figures.

    Args:
        revenue: Total revenue.
        cogs: Cost of goods sold.
        net_income: Net income.
        current_assets: Current assets.
        current_liabilities: Current liabilities.
    """
    gross_margin_pct = round(((revenue - cogs) / revenue) * 100, 2) if revenue else 0.0
    net_margin_pct = round((net_income / revenue) * 100, 2) if revenue else 0.0
    current_ratio = round(current_assets / current_liabilities, 2) if current_liabilities else 0.0
    return (f"gross_margin_pct: {gross_margin_pct}\nnet_margin_pct: {net_margin_pct}\n"
            f"current_ratio: {current_ratio}")

@function_tool
def runway_calculator(cash_on_hand: float, monthly_burn_rate: float) -> str:
    """Compute cash runway in months.

    Args:
        cash_on_hand: Current cash on hand.
        monthly_burn_rate: Average monthly net cash burn.
    """
    if monthly_burn_rate <= 0:
        return "runway_months: infinite (no net burn)"
    return f"runway_months: {round(cash_on_hand / monthly_burn_rate, 1)}"


# --------------------------------------------------
# 6. log_finding — writes into the shared context
# --------------------------------------------------

@function_tool
def log_finding(ctx: RunContextWrapper[Any], agent_name: str, summary: str) -> str:
    """Log a one-line finding to the engagement's audit trail.

    Args:
        agent_name: The name of the agent logging this finding.
        summary: A one-sentence summary of the finding.
    """
    ctx.context.log(agent_name, summary)
    return "Finding logged."


# --------------------------------------------------
# 7. generate_docx_report
# --------------------------------------------------

def _add_inline_markdown(paragraph, text: str) -> None:
    """Split on **bold** / *italic* markers and add each piece as a properly formatted run."""
    import re
    tokens = re.split(r"(\*\*[^*]+\*\*|\*[^*]+\*)", text)
    for tok in tokens:
        if not tok:
            continue
        if tok.startswith("**") and tok.endswith("**"):
            paragraph.add_run(tok[2:-2]).bold = True
        elif tok.startswith("*") and tok.endswith("*"):
            paragraph.add_run(tok[1:-1]).italic = True
        else:
            paragraph.add_run(tok)


def _add_markdown_block(doc, text: str) -> None:
    """Turn an agent's raw Markdown-ish text into real Word paragraphs/tables/bullets
    instead of dumping literal '**', '|', '-' characters onto the page."""
    lines = [ln.rstrip() for ln in str(text).split("\n")]
    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if not line:
            i += 1
            continue

        # Markdown table block: consecutive lines starting with '|'
        if line.startswith("|"):
            table_lines = []
            while i < len(lines) and lines[i].strip().startswith("|"):
                table_lines.append(lines[i].strip())
                i += 1
            # drop separator rows like |---|---|
            rows = [
                [cell.strip() for cell in ln.strip("|").split("|")]
                for ln in table_lines
                if set(ln.replace("|", "").strip()) - set("-: ") != set()
            ]
            if rows:
                n_cols = max(len(r) for r in rows)
                table = doc.add_table(rows=len(rows), cols=n_cols)
                table.style = "Light Grid Accent 1"
                for r, row in enumerate(rows):
                    for col in range(n_cols):
                        cell_text = row[col] if col < len(row) else ""
                        cell_text = cell_text.lstrip("•-* ").strip()
                        p = table.cell(r, col).paragraphs[0]
                        _add_inline_markdown(p, cell_text)
            continue

        # Bullet list line
        if line[:2] in ("- ", "* ", "• ") or line.startswith("\u2022"):
            p = doc.add_paragraph(style="List Bullet")
            _add_inline_markdown(p, line[2:].strip())
            i += 1
            continue

        # Horizontal rule / separator — skip
        if set(line) <= set("-_= "):
            i += 1
            continue

        # Numbered list line, e.g. "1. **Title**"
        import re as _re
        m = _re.match(r"^(\d+)\.\s+(.*)", line)
        if m:
            p = doc.add_paragraph(style="List Number")
            _add_inline_markdown(p, m.group(2))
            i += 1
            continue

        # Plain paragraph
        p = doc.add_paragraph()
        _add_inline_markdown(p, line)
        i += 1


@function_tool
def generate_docx_report(ctx: RunContextWrapper[Any], executive_summary: str) -> str:
    """Generate the final .docx consulting report from the engagement context and an executive summary.

    Args:
        executive_summary: A 3-5 sentence executive summary to place at the top of the report.
    """
    from docx import Document
    c = ctx.context

    doc = Document()
    doc.add_heading(f"Consulting Report — {c.company_name}", level=0)
    doc.add_paragraph(f"Industry: {c.industry}")
    doc.add_paragraph(f"Engagement: {c.raw_problem_statement}")

    doc.add_heading("Executive Summary", level=1)
    _add_markdown_block(doc, executive_summary)

    for heading, content in [
        ("Business Analysis", c.business_analysis),
        ("Market Research", c.market_research),
        ("Competitive Benchmarking", c.benchmark_report),
        ("Financial Analysis", c.financial_analysis),
        ("Strategy & Recommendations", c.strategy),
    ]:
        if content:
            doc.add_heading(heading, level=1)
            _add_markdown_block(doc, content)

    os.makedirs("output", exist_ok=True)
    safe_name = "".join(ch if ch.isalnum() or ch in " _-" else "_" for ch in c.company_name)
    path = os.path.join("output", f"{safe_name}_consulting_report.docx")
    doc.save(path)
    c.report_path = path
    return f"Report saved to {path}"


# --------------------------------------------------
# 8. request_human_approval
# --------------------------------------------------

@function_tool
def request_human_approval(ctx: RunContextWrapper[Any], report_summary: str) -> str:
    """Pause and ask a human to approve the final report before the engagement is marked complete.

    Args:
        report_summary: A short summary of the report to show the human approver.
    """
    c = ctx.context
    if os.environ.get("AUTO_APPROVE", "").lower() == "y":
        c.approval_status = "approved (auto)"
        return "Approved automatically (AUTO_APPROVE=y)."

    display(HTML(f"<div style=\'background:#fef9c3;border-left:4px solid #ca8a04;padding:8px 12px\'>"
                 f"🖐 <b>Human approval requested:</b> {report_summary}</div>"))
    resp = input("Approve this report? (y/n): ").strip().lower()
    c.approval_status = "approved" if resp == "y" else "rejected"
    return f"Human responded: {c.approval_status}"


print("✅ 8 tools defined")

✅ 8 tools defined


## 8. Agent definitions (7 agents)

In [ ]:
from agents import Agent, handoff, RunContextWrapper
from pydantic import BaseModel, Field


class HandoffInput(BaseModel):
    reason: str = Field(
        description="One-sentence reason this engagement is being handed to the Business Analyst"
    )


def on_handoff_to_ba(ctx: RunContextWrapper[ConsultingContext], input_data: HandoffInput) -> None:
    ctx.context.log("Triage Agent", f"Handed off to Business Analyst: {input_data.reason}")


# --------------------------------------------------
# 1. BUSINESS ANALYST
# --------------------------------------------------

business_analyst = Agent[ConsultingContext](
    name="Business Analyst",
    model=openrouter_free_model,
    instructions=(
        "You are a senior business analyst. Read the client's raw problem statement "
        "and turn it into a crisp, structured problem framing. "
        "Use knowledge_base_search to pull relevant facts from uploaded company "
        "documents first. Call log_finding when done. "
        "Finish your reply with a clearly labeled plain-text summary containing "
        "exactly these fields: industry, company_stage, core_question, "
        "key_constraints (as a bullet list), data_gaps (as a bullet list)."
    ),
    tools=[knowledge_base_search, log_finding],
)


# --------------------------------------------------
# 2. MARKET RESEARCH
# --------------------------------------------------

market_research_agent = Agent[ConsultingContext](
    name="Market Research Agent",
    model=openrouter_free_model,
    instructions=(
        "You are a market research analyst. Use web_search to find current data "
        "on market size, growth rate, and trends for the given industry. "
        "Call log_finding before finishing. "
        "Finish with a labeled summary containing: market_size_summary, "
        "growth_trend, key_drivers (bullet list), risks (bullet list), "
        "sources (bullet list of URLs/citations)."
    ),
    tools=[web_search, log_finding],
)


# --------------------------------------------------
# 3. BENCHMARKING
# --------------------------------------------------

benchmarking_agent = Agent[ConsultingContext](
    name="Benchmarking Agent",
    model=groq_heavy_model,
    instructions=(
        "You are a competitive benchmarking specialist. "
        "Call competitor_benchmark_lookup first, then use web_search "
        "to identify 2-4 REAL, named competitors. "
        "Call log_finding before finishing. "
        "Finish with a labeled summary: for each competitor list name, "
        "positioning, strengths (bullet list), weaknesses (bullet list); "
        "then company_relative_position."
    ),
    tools=[competitor_benchmark_lookup, web_search, log_finding],
)


# --------------------------------------------------
# 4. FINANCIAL ANALYSIS
# --------------------------------------------------

financial_analysis_agent = Agent[ConsultingContext](
    name="Financial Analysis Agent",
    model=gemini_model,
    instructions=(
        "You are a financial analyst. Use financial_ratio_calculator "
        "and runway_calculator on the figures provided. "
        "Never invent numbers the user didn\'t supply. "
        "Call log_finding before finishing. "
        "Finish with a labeled summary: gross_margin_pct, net_margin_pct, "
        "current_ratio, runway_months, headline_assessment."
    ),
    tools=[financial_ratio_calculator, runway_calculator, log_finding],
)


# --------------------------------------------------
# 5. STRATEGY ADVISOR
# --------------------------------------------------

strategy_advisor = Agent[ConsultingContext](
    name="Strategy Advisor",
    model=gemini_model,
    instructions=(
        "You are a senior strategy consultant. Synthesize the business analysis, "
        "market research, benchmarking, and financial analysis into a SWOT "
        "and 3-5 prioritized recommendations. "
        "REFLECTION STEP: before finalizing, re-read your own draft and check "
        "every recommendation against the data you were given. "
        "Call log_finding before finishing. "
        "Finish with a labeled summary: swot (list of category + point pairs), "
        "recommendations (list of title, rationale, priority, expected_impact), "
        "self_review_notes."
    ),
    tools=[log_finding],
)


# --------------------------------------------------
# 6. REPORT WRITER
# --------------------------------------------------

report_writer = Agent[ConsultingContext](
    name="Report Writer",
    model=groq_fast_model,
    instructions=(
        "You are a consulting report writer. Write a tight 3-5 sentence "
        "executive summary based on the context, call generate_docx_report "
        "with it, then call request_human_approval with a short report "
        "summary so a human can sign off. "
        "Report back the file path and approval decision."
    ),
    tools=[generate_docx_report, request_human_approval],
)


# --------------------------------------------------
# 7. TRIAGE AGENT
# --------------------------------------------------

triage_agent = Agent[ConsultingContext](
    name="Triage Agent",
    model=gemini_model,  # small/fast Groq models were unreliable here, hallucinating a nonexistent tool
    instructions=(
        "You are the front door of an AI management consulting firm. "
        "You have exactly one available action: call the transfer_to_business_analyst "
        "function with a one-sentence reason for the handoff. Do not call any other "
        "function or tool, and do not attempt the analysis yourself. Call "
        "transfer_to_business_analyst immediately."
    ),
    handoffs=[
        handoff(
            business_analyst,
            input_type=HandoffInput,
            on_handoff=on_handoff_to_ba,
        )
    ],
)


print(
    "✅ 7 agents defined "
    "(Triage, Business Analyst, Market Research, Benchmarking, "
    "Financial Analysis, Strategy Advisor, Report Writer)"
)

✅ 7 agents defined (Triage, Business Analyst, Market Research, Benchmarking, Financial Analysis, Strategy Advisor, Report Writer)


## 9. Orchestration — handoff → parallel research → strategy → report

Each stage prints its own progress panel and result right below this cell when the engagement runs.

**No shared session is used.** Each stage's prompt is built explicitly from `ConsultingContext`, so
input tokens stay proportional to that one stage's own work instead of the whole engagement's
growing transcript — this is what actually fixes free-tier quota exhaustion (see the intro cell).

In [ ]:
from agents import Runner
from agents.exceptions import AgentsException, MaxTurnsExceeded

async def run_stage(agent, prompt, context, stage_name, color="#2563eb"):
    show(stage_name, color=color)
    try:
        result = await Runner.run(agent, prompt, context=context, max_turns=8)
        text = result.final_output if isinstance(result.final_output, str) else str(result.final_output)
        show(f"{stage_name} — result", text, color="#16a34a")
        return text
    except MaxTurnsExceeded:
        show_error(stage_name, "exceeded max turns")
        context.log(stage_name, "FAILED: exceeded max turns")
    except AgentsException as e:
        show_error(stage_name, str(e))
        context.log(stage_name, f"FAILED: {e}")
    except Exception as e:
        show_error(stage_name, str(e))
        context.log(stage_name, f"FAILED (unexpected): {e}")
    return None


async def run_engagement(client_id, company_name, industry, problem_statement, financial_figures):
    context = ConsultingContext(client_id=client_id, company_name=company_name, industry=industry,
                                 raw_problem_statement=problem_statement, financial_figures=financial_figures)

    intake_prompt = (f"Client: {company_name}\nIndustry: {industry}\nProblem statement: {problem_statement}\n"
                      f"Financial figures supplied: {financial_figures}")
    context.business_analysis = await run_stage(triage_agent, intake_prompt, context, "1️⃣ Triage + Business Analysis")

    display(HTML("<h3 style=\'color:#7c3aed\'>2️⃣ Parallel: Market Research | Benchmarking | Financial Analysis</h3>"))
    mr_prompt = f"Industry: {industry}. Company: {company_name}. Context: {problem_statement}"
    bm_prompt = f"Company: {company_name}. Industry: {industry}."
    fin_prompt = f"Financial figures: {financial_figures}"

    context.market_research, context.benchmark_report, context.financial_analysis = await asyncio.gather(
        run_stage(market_research_agent, mr_prompt, context, "  📈 Market Research", "#7c3aed"),
        run_stage(benchmarking_agent, bm_prompt, context, "  🏁 Benchmarking", "#7c3aed"),
        run_stage(financial_analysis_agent, fin_prompt, context, "  💰 Financial Analysis", "#7c3aed"),
    )

    strategy_prompt = (f"Synthesize the following into a SWOT and prioritized recommendations.\n"
                        f"Business analysis: {context.business_analysis}\nMarket research: {context.market_research}\n"
                        f"Benchmarking: {context.benchmark_report}\nFinancial analysis: {context.financial_analysis}")
    context.strategy = await run_stage(strategy_advisor, strategy_prompt, context, "3️⃣ Strategy Synthesis (+ self-review)", "#dc2626")

    report_prompt = (f"Write the final consulting report and request human approval.\n\n"
                      f"Client: {company_name} ({industry})\nProblem: {problem_statement}\n\n"
                      f"Business analysis: {context.business_analysis}\nMarket research: {context.market_research}\n"
                      f"Benchmarking: {context.benchmark_report}\nFinancial analysis: {context.financial_analysis}\n"
                      f"Strategy: {context.strategy}")
    await run_stage(report_writer, report_prompt, context, "4️⃣ Report Writing + Human Approval", "#ea580c")

    return context

print("✅ Orchestration ready (session-free, token-lean)")

✅ Orchestration ready (session-free, token-lean)


## 10. Run it 🚀
Edit the arguments below for your own business problem, or run as-is for the demo (fictional bakery).

Set `AUTO_APPROVE=y` in the next cell if you want to skip the interactive approval prompt.

In [ ]:
# os.environ["AUTO_APPROVE"] = "y"   # uncomment to skip the interactive approval prompt

figures = {
    "revenue": 1_200_000, "cogs": 480_000, "net_income": 90_000,
    "current_assets": 350_000, "current_liabilities": 210_000,
    "cash_on_hand": 400_000, "monthly_burn_rate": 45_000,
}

context = asyncio.run(run_engagement(
    client_id="demo-client-001",
    company_name="Northwind Bakery Co",
    industry="artisan bakery / specialty food retail",
    problem_statement="We want to expand from 1 retail location to a regional wholesale + e-commerce model within 18 months. Should we, and how?",
    financial_figures=figures,
))

display(HTML("<h2 style=\'color:#16a34a\'>✅ Engagement Complete</h2>"))
display(Markdown(f"**Report path:** `{context.report_path}`  \n**Approval status:** `{context.approval_status}`"))

Here's a labeled summary for each competitor:
1. Tartine Bakery: 
- Name: Tartine Bakery
- Positioning: Upscale artisan bakery with a focus on high-quality, unique ingredients
- Strengths: 
    * High-quality ingredients
    * Unique flavor combinations
    * Strong brand reputation
- Weaknesses: 
    * Limited geographical presence
    * High prices may deter budget-conscious customers
- Company relative position: Northwind Bakery Co. can differentiate itself by offering more affordable prices without compromising on quality.

2. IZZIO ARTISAN BAKERY: 
- Name: IZZIO ARTISAN BAKERY
- Positioning: Artisan bakery with a focus on traditional baking methods and high-quality ingredients
- Strengths: 
    * Traditional baking methods
    * High-quality ingredients
    * Strong brand reputation
- Weaknesses: 
    * Limited product offerings
    * High prices may deter budget-conscious customers
- Company relative position: Northwind Bakery Co. can expand its product offerings to cater to a wider range of customers.

3. Bakery Bakery: 
- Name: Bakery Bakery
- Positioning: Bakery with a focus on traditional baked goods and a wide range of products
- Strengths: 
    * Wide range of products
    * Traditional baking methods
    * Affordable prices
- Weaknesses: 
    * Limited geographical presence
    * Quality may vary depending on location
- Company relative position: Northwind Bakery Co. can focus on maintaining consistent quality across all its products and locations.

4. Cinnabon: 
- Name: Cinnabon
- Positioning: Bakery with a focus on sweet, indulgent treats
- Strengths: 
    * Unique and indulgent products
    * Strong brand reputation
    * Wide geographical presence
- Weaknesses: 
    * Limited product offerings
    * High calorie counts may deter health-conscious customers
- Company relative position: Northwind Bakery Co. can offer healthier alternatives to Cinnabon's products while maintaining the same level of indulgence.

By analyzing these competitors, Northwind Bakery Co. can refine its strategy to differentiate itself, expand its product offerings, and maintain consistent quality to attract and retain customers.

## 11. Download / open the report

In [ ]:
from IPython.display import FileLink

if context.report_path and os.path.exists(context.report_path):
    try:
        from google.colab import files
        files.download(context.report_path)   # triggers a browser download in Colab
    except ImportError:
        display(FileLink(context.report_path))  # clickable link in VS Code / Jupyter
else:
    print("No report was generated — check the FAILED entries above.")

## 12. Audit trail (all findings logged across agents)

In [ ]:
for entry in context.findings_log:
    display(Markdown(f"- **[{entry['agent']}]** {entry['summary']}"))